# Q3 — Feature Engineering and Regression Pipeline
**Goal:** Predict `items_sold` at retail stores using a scikit-learn pipeline.
**Dataset:** `q3_retail_promotions.csv` (1200 rows, 9 columns)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded successfully!')

## Task 1 — Date Feature Engineering (4 marks)

In [ ]:
df = pd.read_csv('../data/q3_retail_promotions.csv')
print('Shape:', df.shape)
print('\nFirst 3 rows:')
print(df.head(3))

# Parse transaction_date
df['transaction_date'] = pd.to_datetime(df['transaction_date'])

# Extract date features
df['year']         = df['transaction_date'].dt.year
df['month']        = df['transaction_date'].dt.month
df['day_of_week']  = df['transaction_date'].dt.dayofweek

# Binary feature: is_month_end (1 if day of month >= 25)
df['is_month_end'] = (df['transaction_date'].dt.day >= 25).astype(int)

print('\nSample with new date features:')
print(df[['transaction_date', 'year', 'month', 'day_of_week',
          'is_month_end', 'items_sold']].head(8))

## Task 2 — Temporal Train-Test Split (3 marks)

In [ ]:
# Sort by date and use most recent 20% as test set
df_sorted   = df.sort_values('transaction_date').reset_index(drop=True)
split_index = int(len(df_sorted) * 0.8)

train_df = df_sorted.iloc[:split_index]
test_df  = df_sorted.iloc[split_index:]

print(f'Training set: {len(train_df)} rows')
print(f'  Date range: {train_df["transaction_date"].min().date()} '
      f'to {train_df["transaction_date"].max().date()}')
print(f'Test set    : {len(test_df)} rows')
print(f'  Date range: {test_df["transaction_date"].min().date()} '
      f'to {test_df["transaction_date"].max().date()}')

**Why random split is inappropriate for time-ordered data:** A random split would allow the model to train on future data and test on past data, creating data leakage. For example, if the model sees December sales during training but predicts October sales during testing, it effectively uses future information to predict the past — this produces unrealistically optimistic evaluation metrics. In production, a model always predicts the future from the past. A temporal split respects this direction of time and gives a realistic assessment of how the model will perform on genuinely unseen future records.

## Task 3 — Preprocessing Pipeline (5 marks)

In [ ]:
# Define feature columns
categorical_features = ['promotion_type', 'location_type', 'store_size']
numerical_features   = ['store_id', 'is_weekend', 'is_festival',
                         'competition_density', 'year', 'month',
                         'day_of_week', 'is_month_end']
target = 'items_sold'

feature_cols = categorical_features + numerical_features

X_train = train_df[feature_cols]
y_train = train_df[target]
X_test  = test_df[feature_cols]
y_test  = test_df[target]

# Build ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('ohe',    OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ('scaler', StandardScaler(),                       numerical_features)
])

print('Preprocessing pipeline built!')
print(f'Categorical features: {categorical_features}')
print(f'Numerical features  : {numerical_features}')

## Task 4 — Model Training and Evaluation (8 marks)

In [ ]:
# Model 1: Linear Regression Pipeline
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model',        LinearRegression())
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr  = mean_absolute_error(y_test, y_pred_lr)

print('Linear Regression:')
print(f'  RMSE: {rmse_lr:.4f}')
print(f'  MAE : {mae_lr:.4f}')

# Model 2: Random Forest Regressor Pipeline
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model',        RandomForestRegressor(random_state=42, n_estimators=100))
])

rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf  = mean_absolute_error(y_test, y_pred_rf)

print('\nRandom Forest Regressor:')
print(f'  RMSE: {rmse_rf:.4f}')
print(f'  MAE : {mae_rf:.4f}')

In [ ]:
# Parity plots (predicted vs actual) for both models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, title in zip(
    axes,
    [y_pred_lr, y_pred_rf],
    ['Linear Regression', 'Random Forest']
):
    ax.scatter(y_test, y_pred, alpha=0.5, color='steelblue', edgecolors='black', s=30)
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val],
            color='red', linestyle='--', linewidth=1.5, label='Perfect Prediction')
    ax.set_title(f'Parity Plot — {title}')
    ax.set_xlabel('Actual Items Sold')
    ax.set_ylabel('Predicted Items Sold')
    ax.legend()

plt.tight_layout()
plt.savefig('plot_parity.png')
plt.show()
print('plot_parity.png saved.')

In [ ]:
# Feature importances from Random Forest
ohe_features = list(
    rf_pipeline.named_steps['preprocessor']
    .named_transformers_['ohe']
    .get_feature_names_out(categorical_features)
)
all_features     = ohe_features + numerical_features
importances      = rf_pipeline.named_steps['model'].feature_importances_
importance_df    = pd.DataFrame({'feature': all_features,
                                  'importance': importances})
importance_df    = importance_df.sort_values('importance', ascending=False)

print('Feature Importances (Random Forest):')
print(importance_df.head(10).to_string(index=False))

print('\nTop 5 most influential features:')
for i, row in importance_df.head(5).iterrows():
    print(f'  {row["feature"]}: {row["importance"]:.4f}')

# Plot top 10 feature importances
plt.figure(figsize=(10, 6))
top10 = importance_df.head(10)
plt.barh(top10['feature'][::-1], top10['importance'][::-1],
         color='steelblue', edgecolor='black')
plt.title('Top 10 Feature Importances — Random Forest Regressor')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig('plot_feature_importance.png')
plt.show()
print('plot_feature_importance.png saved.')